# Violence Detection — Kaggle runner

This notebook fetches the repository and runs its version-controlled smoke-test script. Add `WANDB_API_KEY` as a Kaggle Secret for online tracking. Add `GITHUB_TOKEN` only when the repository is private. Never paste either secret into a cell.


## 1. Kaggle secrets and runtime


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ayaan-Ali-Khan/Violence-Detection.git"
REPO_DIR = Path("/kaggle/working/Violence-Detection")
GIT_REF = "main"  # Change to a branch, tag, or commit SHA to pin a version.

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
except Exception as exc:
    print(f"WANDB_API_KEY unavailable; logger will use its local fallback: {exc}")

try:
    github_token = secrets.get_secret("GITHUB_TOKEN")
except Exception:
    github_token = None

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Clone or update the selected revision


In [ ]:
env = os.environ.copy()
if github_token:
    # Use a temporary Git config header; the token is not placed in the URL or printed.
    import base64
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"], check=True, env=env)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_REF], check=True, env=env)
subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=True)


## 3. Install project dependencies

Kaggle's preinstalled CUDA-enabled PyTorch is retained. The project is installed without resolving dependencies, then the Kaggle-specific list installs the non-Torch requirements.


In [ ]:
%cd /kaggle/working/Violence-Detection
!python -m pip install -e . --no-deps
!python -m pip install -r requirements-kaggle.txt


## 4. Run the integration smoke test


In [ ]:
!python scripts/smoke_test.py --config configs/smoke.yaml


## 5. Keep the result with the Kaggle output

The smoke-test JSON and TensorBoard fallback logs are written under `/kaggle/working/violence-detection-output`. Use Kaggle's **Save Version** action to preserve notebook outputs.


In [ ]:
output_dir = Path("/kaggle/working/violence-detection-output")
print("Output files:")
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_dir), path.stat().st_size, "bytes")
